# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
metadata_json = metadata.to_json()

print(f"Dataset Title: {metadata_json.get('name', '')}\nDescription: {metadata_json.get('description', '')}\n")


## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we inspect the available record sets (tables) and their schema using their `@id`s, fields, and columns. All references use the records set, field, and column `@id` for clarity and future reproducibility.

_Note: If you are exploring a new dataset, the structure may be printed for your guidance._


In [ ]:
# List all available record sets and their IDs
print("Available record sets and fields (using @id):\n")
for record_set in metadata.record_sets:
    print(f"RecordSet name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {record_set.description}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Example below extracts all available dataframes for all record sets found. You can select a record set for further analysis.**


In [ ]:
# Extract all available record sets' @id
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Print columns for each extracted DataFrame
for record_set_id in dataframes:
    print(f"Columns for record set {record_set_id}:\n  {dataframes[record_set_id].columns.tolist()}\n")

# Preview the first record set
if len(record_set_ids) > 0:
    print(f"Preview of '{record_set_ids[0]}' records:")
    display(dataframes[record_set_ids[0]].head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. For demonstration, we select the first (main) record set and a numeric field if found in its schema.


In [ ]:
# Choose a record set and some numeric field for analysis
selected_record_set_id = record_set_ids[0]  # Main table, feel free to choose as needed
df = dataframes[selected_record_set_id]

# Find a numeric (Float or Integer) field @id from metadata
numeric_field_id = None
group_field_id = None
for field in metadata.record_sets[0].fields:
    if field.data_type in ('schema:Float', 'schema:Integer') and (field.id in df.columns):
        numeric_field_id = field.id
        break

# As example, also pick a non-numeric field for optional grouping
for field in metadata.record_sets[0].fields:
    # Pick first non-numeric field for grouping
    if field.data_type == 'schema:Text' and field.id in df.columns:
        group_field_id = field.id
        break

if numeric_field_id is not None:
    # Drop rows where numeric field is NaN
    numeric_df = df.copy()
    numeric_df = numeric_df[pd.to_numeric(numeric_df[numeric_field_id], errors='coerce').notnull()]
    numeric_df[numeric_field_id] = pd.to_numeric(numeric_df[numeric_field_id], errors='coerce')
    threshold = numeric_df[numeric_field_id].mean()  # Example: use mean as threshold
    filtered_df = numeric_df[numeric_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optionally group by a categorical field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field detected in the first record set for EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For demonstration, we plot a histogram of the selected numeric field (if available), and a boxplot grouped by a categorical field (if such field exists).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field found for visualization.')


## 6. Conclusion
In this notebook, we demonstrated how to load, explore, preprocess, and visualize the FAIR^2 dataset on mass spectrometry analysis of extracellular vesicles using the `mlcroissant` library. All dataset entities, such as record sets and fields, were referenced by their stable `@id` per the Croissant schema specification. For further exploration, refer to the available field and column IDs printed in section 2.
